# ∫ Calculus for Machine Learning — Hands-On

Train models from first principles. This notebook builds from derivatives to backpropagation to modern optimisers — all with intuitive visualisations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
print('Ready!')

---
## 1. Derivatives from First Principles

In [ ]:
# f'(a) = lim_{h→0} [f(a+h) - f(a)] / h
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)  # central difference (more accurate)

# Test on x^3
f  = lambda x: x**3
df = lambda x: 3*x**2  # analytical

x_vals = np.linspace(-2, 2, 200)
h_values = [0.5, 0.1, 0.01, 1e-5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_vals, f(x_vals), 'k-', lw=2, label='f(x) = x³')
a = 1.0  # point to differentiate at
axes[0].plot(a, f(a), 'ro', markersize=10, label=f'Point a={a}')
# Draw tangent lines for different h
for h, col in zip(h_values[:3], ['red', 'orange', 'green']):
    slope = (f(a + h) - f(a)) / h  # forward difference
    x_tan = np.linspace(a-0.8, a+0.8, 100)
    axes[0].plot(x_tan, f(a) + slope * (x_tan - a), color=col, alpha=0.7, label=f'h={h}, slope={slope:.3f}')
axes[0].plot(x_vals, df(1) + df(1)*0 + df(x_vals)*0 + df(1)*(x_vals-1), 'b--', label=f'True tangent: slope={df(a)}')
axes[0].set_xlim(0, 2); axes[0].set_ylim(-1, 8)
axes[0].legend(fontsize=8)
axes[0].set_title("Derivative = Slope of Tangent Line\n(Secant lines converge as h→0)")

# Convergence of numerical derivative
h_range = np.logspace(-10, 0, 100)
errors = [abs(numerical_derivative(f, 1.0, h) - df(1.0)) for h in h_range]
axes[1].loglog(h_range, errors, 'b-', lw=2)
axes[1].set_xlabel('Step size h'); axes[1].set_ylabel('|Numerical - Analytical|')
axes[1].set_title('Error in Numerical Derivative vs Step Size\n(Too small h → floating point noise)')

plt.tight_layout()
plt.show()

print(f"Analytical f'(1) = {df(1.0)}")
print(f"Numerical  f'(1) = {numerical_derivative(f, 1.0):.10f}")

---
## 2. Activation Functions & Their Derivatives

In [ ]:
x = np.linspace(-4, 4, 500)

activations = {
    'Sigmoid':    (lambda x: 1/(1+np.exp(-x)),
                   lambda x: (1/(1+np.exp(-x))) * (1 - 1/(1+np.exp(-x)))),
    'Tanh':       (np.tanh,
                   lambda x: 1 - np.tanh(x)**2),
    'ReLU':       (lambda x: np.maximum(0, x),
                   lambda x: (x > 0).astype(float)),
    'Leaky ReLU': (lambda x: np.where(x > 0, x, 0.01*x),
                   lambda x: np.where(x > 0, 1.0, 0.01)),
    'GELU':       (lambda x: x * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi)*(x + 0.044715*x**3))),
                   lambda x: numerical_derivative(lambda t: t * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi)*(t + 0.044715*t**3))), x)),
    'Swish':      (lambda x: x * (1/(1+np.exp(-x))),
                   lambda x: (1/(1+np.exp(-x))) + x*(1/(1+np.exp(-x)))*(1 - 1/(1+np.exp(-x)))),
}

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
colors = ['blue','darkorange','green','red','purple','brown']

for col_idx, (name, (f, df_fn)) in enumerate(activations.items()):
    axes[0, col_idx].plot(x, f(x), color=colors[col_idx], lw=2)
    axes[0, col_idx].set_title(f'{name}')
    axes[0, col_idx].set_xlabel('x')
    if col_idx == 0: axes[0, col_idx].set_ylabel('f(x)')
    axes[0, col_idx].set_ylim(-2, 2)

    axes[1, col_idx].plot(x, df_fn(x), color=colors[col_idx], lw=2, linestyle='--')
    axes[1, col_idx].set_title(f"f'(x) — gradient")
    axes[1, col_idx].set_xlabel('x')
    if col_idx == 0: axes[1, col_idx].set_ylabel("f'(x)")
    axes[1, col_idx].set_ylim(-0.5, 1.5)

plt.suptitle('Activation Functions & Their Derivatives\n(derivative = gradient signal during backprop)', fontsize=14)
plt.tight_layout()
plt.show()

print("Key observations:")
print("Sigmoid: max gradient = 0.25  → gradients shrink by ≥4× per layer (vanishing!)")
print("ReLU:    gradient = 0 for x<0 → dead neurons (never update)")
print("GELU:    smooth, non-zero gradient everywhere → used in GPT, BERT, etc.")

---
## 3. The Chain Rule — Step-by-Step

In [ ]:
# Manual chain rule for a simple computation graph
# Forward: z = (x + y) * (x - y)  — two operations

x_val, y_val = 3.0, 1.0

# Forward pass (cache intermediates)
a = x_val + y_val   # a = 4
b = x_val - y_val   # b = 2
z = a * b           # z = 8

print("=== Forward Pass ===")
print(f"a = x + y = {a}")
print(f"b = x - y = {b}")
print(f"z = a * b = {z}")

# Backward pass (chain rule from output to inputs)
dz_dz = 1.0             # gradient of z w.r.t. itself
dz_da = b               # ∂(a*b)/∂a = b
dz_db = a               # ∂(a*b)/∂b = a
da_dx = 1.0             # ∂(x+y)/∂x = 1
da_dy = 1.0             # ∂(x+y)/∂y = 1
db_dx = 1.0             # ∂(x-y)/∂x = 1
db_dy = -1.0            # ∂(x-y)/∂y = -1

# Chain rule: ∂z/∂x = (∂z/∂a)(∂a/∂x) + (∂z/∂b)(∂b/∂x)
dz_dx = dz_da * da_dx + dz_db * db_dx
dz_dy = dz_da * da_dy + dz_db * db_dy

print("\n=== Backward Pass (Chain Rule) ===")
print(f"∂z/∂a = b = {dz_da}")
print(f"∂z/∂b = a = {dz_db}")
print(f"∂z/∂x = (∂z/∂a)(∂a/∂x) + (∂z/∂b)(∂b/∂x) = {dz_da}*{da_dx} + {dz_db}*{db_dx} = {dz_dx}")
print(f"∂z/∂y = (∂z/∂a)(∂a/∂y) + (∂z/∂b)(∂b/∂y) = {dz_da}*{da_dy} + {dz_db}*{db_dy} = {dz_dy}")

# Analytical verification: z = (x+y)(x-y) = x² - y²
# ∂z/∂x = 2x = 6,  ∂z/∂y = -2y = -2
print(f"\nAnalytical: ∂z/∂x = 2x = {2*x_val}  ✓")
print(f"Analytical: ∂z/∂y = -2y = {-2*y_val}  ✓")

# Now use PyTorch autograd
x_t = torch.tensor(3.0, requires_grad=True)
y_t = torch.tensor(1.0, requires_grad=True)
z_t = (x_t + y_t) * (x_t - y_t)
z_t.backward()
print(f"\nPyTorch autograd: ∂z/∂x = {x_t.grad.item()}, ∂z/∂y = {y_t.grad.item()}  ✓")

---
## 4. Gradients & Partial Derivatives — Visualised

In [ ]:
# Visualise a 2D function and its gradient field
# f(x,y) = x² + 2y²  (an elliptic paraboloid — a bowl)

f2d  = lambda x, y: x**2 + 2*y**2
df_x = lambda x, y: 2*x
df_y = lambda x, y: 4*y

x_g = np.linspace(-3, 3, 200)
y_g = np.linspace(-3, 3, 200)
X_g, Y_g = np.meshgrid(x_g, y_g)
Z_g = f2d(X_g, Y_g)

# Gradient field
xq = np.linspace(-2.5, 2.5, 12)
yq = np.linspace(-2.5, 2.5, 12)
Xq, Yq = np.meshgrid(xq, yq)
Gx = df_x(Xq, Yq)
Gy = df_y(Xq, Yq)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contour plot + gradient arrows
cs = axes[0].contourf(X_g, Y_g, Z_g, levels=20, cmap='viridis')
plt.colorbar(cs, ax=axes[0])
axes[0].quiver(Xq, Yq, Gx, Gy, alpha=0.7, color='white', scale=100)
axes[0].set_title('f(x,y) = x² + 2y²\n(gradient arrows point uphill — we go the OPPOSITE way)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# 3D surface
axes[1] = fig.add_subplot(1, 2, 2, projection='3d')
axes[1].plot_surface(X_g[::5,::5], Y_g[::5,::5], Z_g[::5,::5], cmap='viridis', alpha=0.7)
axes[1].set_xlabel('x'); axes[1].set_ylabel('y'); axes[1].set_zlabel('f(x,y)')
axes[1].set_title('3D Loss Landscape')

plt.tight_layout()
plt.show()

print("Gradient at (1, 1): ∇f = [∂f/∂x, ∂f/∂y] = [{}, {}]".format(df_x(1,1), df_y(1,1)))
print("Notice: gradient in y-direction is larger because the bowl is steeper in y (coefficient 2)")
print("This is why features with different scales make gradient descent zig-zag → normalise your data!")

---
## 5. Gradient Descent — Built from Scratch

In [ ]:
# Compare GD on: (1) normalised vs unnormalised features
# Loss = (2x)² + (0.1y)²  (very different scales)

def gd_path(f, grad_f, start, lr, steps=100):
    path = [start.copy()]
    p = start.copy()
    for _ in range(steps):
        p = p - lr * grad_f(p)
        path.append(p.copy())
    return np.array(path)

# Ill-conditioned loss (features at different scales)
f_bad    = lambda p: (2*p[0])**2 + (0.1*p[1])**2
grad_bad = lambda p: np.array([8*p[0], 0.02*p[1]])

# Well-conditioned (after normalising)
f_good    = lambda p: p[0]**2 + p[1]**2
grad_good = lambda p: 2*p

start = np.array([2.5, 2.5])
path_bad  = gd_path(f_bad, grad_bad, start.copy(), lr=0.05, steps=50)
path_good = gd_path(f_good, grad_good, start.copy(), lr=0.3, steps=50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, f_fn, path, title in zip(axes, [f_bad, f_good], [path_bad, path_good],
                                  ['Ill-conditioned (different feature scales)\nzig-zag trajectory',
                                   'Well-conditioned (normalised features)\nstraight to minimum']):
    X1 = np.linspace(-3, 3, 100); X2 = np.linspace(-3, 3, 100)
    XX1, XX2 = np.meshgrid(X1, X2)
    ZZ = np.array([[f_fn(np.array([x1, x2])) for x1 in X1] for x2 in X2])
    ax.contourf(XX1, XX2, ZZ, levels=20, cmap='RdYlGn_r', alpha=0.6)
    ax.plot(path[:, 0], path[:, 1], 'b.-', lw=1.5, markersize=6, label='GD path')
    ax.plot(*start, 'gs', markersize=12, label='Start')
    ax.plot(0, 0, 'r*', markersize=15, label='Minimum')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# SGD, Momentum, and Adam — compare convergence on a noisy loss
np.random.seed(42)

# Simple 1D loss for clarity: f(x) = x^2 (minimum at 0)
# But gradients are noisy (simulating mini-batch SGD)
def noisy_grad(x, noise_std=0.5):
    return 2*x + np.random.normal(0, noise_std)

def run_optimiser(name, n_steps=200, lr=0.1, noise=0.5):
    x = 5.0
    m, v = 0.0, 0.0
    beta1, beta2, eps = 0.9, 0.999, 1e-8
    xs = [x]
    
    for t in range(1, n_steps+1):
        g = noisy_grad(x, noise)
        
        if name == 'SGD':
            x = x - lr * g
        elif name == 'Momentum':
            m = 0.9 * m + (1 - 0.9) * g
            x = x - lr * m
        elif name == 'Adam':
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g**2
            m_hat = m / (1 - beta1**t)
            v_hat = v / (1 - beta2**t)
            x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        
        xs.append(x)
    return np.array(xs)

optimisers = {'SGD': 0.1, 'Momentum': 0.1, 'Adam': 0.5}
colors = {'SGD': 'blue', 'Momentum': 'darkorange', 'Adam': 'green'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, lr in optimisers.items():
    xs = run_optimiser(name, lr=lr)
    axes[0].plot(xs, alpha=0.8, color=colors[name], label=name, lw=1.5)
    axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
    axes[1].plot(xs**2, alpha=0.8, color=colors[name], label=name, lw=1.5)  # loss = x^2

axes[0].set_title('Parameter x over Time'); axes[0].legend()
axes[0].set_xlabel('Step'); axes[0].set_ylabel('x value')
axes[1].set_title('Loss = x² over Time'); axes[1].legend()
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("💡 Adam converges fastest by adapting the LR per parameter.")
print("   SGD with momentum is still competitive and often generalises better.")

---
## 6. Loss Landscapes — Saddle Points, Minima, Convexity

In [ ]:
# Visualise different types of critical points
fig = plt.figure(figsize=(18, 6))

x1 = np.linspace(-2, 2, 100)
x2 = np.linspace(-2, 2, 100)
X1, X2 = np.meshgrid(x1, x2)

surfaces = [
    (X1**2 + X2**2,         'Local/Global Min\nx²+y²  (convex)',     '🟢'),
    (X1**2 - X2**2,         'Saddle Point\nx²-y²  (non-convex)',     '🟡'),
    (0.5*X1**4 - X1**2 + X2**2, 'Multiple Minima\n0.5x⁴-x²+y²',  '🔴'),
]

for i, (Z, title, emoji) in enumerate(surfaces):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    ax.plot_surface(X1, X2, Z, cmap='coolwarm', alpha=0.8, linewidth=0)
    ax.set_title(f'{emoji} {title}', fontsize=11)
    ax.set_xlabel('θ₁'); ax.set_ylabel('θ₂'); ax.set_zlabel('Loss')
    ax.view_init(30, 45)

plt.suptitle('Types of Critical Points in Loss Landscapes', fontsize=14)
plt.tight_layout()
plt.show()

# Gradient at the saddle point (0,0) of x²-y²: should be [0,0]
f_saddle = lambda x, y: x**2 - y**2
print("At saddle point (0,0) of x²-y²:")
print(f"  ∂f/∂x = 2x = 0  ✓")
print(f"  ∂f/∂y = -2y = 0  ✓")
print("Gradient = 0 → GD gets STUCK. But SGD noise helps escape!")

# Check with Hessian
H = np.array([[2, 0], [0, -2]])
eigenvals = np.linalg.eigvals(H)
print(f"\nHessian eigenvalues: {eigenvals}")
print("Mixed signs → confirmed saddle point")

In [ ]:
# Convexity test & learning rate effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convex vs non-convex
x = np.linspace(-2, 2, 300)
axes[0].plot(x, x**2, 'b-', lw=2, label='x² (convex)')
axes[0].plot(x, x**4 - 2*x**2, 'r-', lw=2, label='x⁴ - 2x² (non-convex, 2 minima)')
# Show convexity: midpoint of chord lies above function for convex
x1_c, x2_c = -1.5, 1.5
axes[0].plot([x1_c, x2_c], [x1_c**2, x2_c**2], 'b--', alpha=0.5, label='Chord (above → convex)')
axes[0].set_title('Convex vs Non-Convex Functions')
axes[0].set_xlabel('x'); axes[0].legend()

# Learning rate effect
def gd_1d(lr, steps=30):
    x = 3.0
    xs = [x]
    for _ in range(steps):
        x = x - lr * 2 * x  # gradient of x^2
        xs.append(x)
    return np.array(xs)

lrs = [0.01, 0.1, 0.5, 1.0, 1.1]
colors_lr = ['navy', 'steelblue', 'green', 'orange', 'red']
labels_lr = ['LR=0.01 (slow)', 'LR=0.1 (good)', 'LR=0.5 (good)', 'LR=1.0 (oscillates)', 'LR=1.1 (diverges!)']

for lr, col, label in zip(lrs, colors_lr, labels_lr):
    xs = gd_1d(lr)
    losses = xs**2
    axes[1].plot(losses, color=col, lw=2, label=label)
axes[1].axhline(0, color='k', linestyle='--', label='Optimum')
axes[1].set_yscale('symlog')
axes[1].set_title('Effect of Learning Rate on Convergence\n(Minimising x²)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss (log scale)')
axes[1].legend(fontsize=8)
axes[1].set_ylim(-1, 100)

plt.tight_layout()
plt.show()

print("For f(x)=x², gradient = 2x, so GD: x ← x - lr*2x = x(1-2*lr)")
print("Converges when |1-2*lr| < 1, i.e., lr < 1.0")
print("For lr=1.0: x(1-2) = -x → oscillates. For lr>1: diverges.")

---
## 7. Backpropagation — Full Implementation from Scratch

In [ ]:
# Full backprop for a 2-layer neural net — no libraries
# Network: input(2) → hidden(4) → output(1) → Binary classification

np.random.seed(0)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(a):
    return a * (1 - a)  # a is already sigmoid(z)

def bce(y, p, eps=1e-9):
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

# Generate XOR dataset (non-linearly separable)
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)  # XOR labels

# Initialise weights (He init)
n_hidden = 4
W1 = np.random.randn(2, n_hidden) * np.sqrt(2/2)
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, 1) * np.sqrt(2/n_hidden)
b2 = np.zeros((1, 1))

lr = 0.5
losses_bp = []

for epoch in range(5000):
    # ── FORWARD PASS ──
    Z1 = X_xor @ W1 + b1          # (4, 4)
    A1 = sigmoid(Z1)               # (4, 4)
    Z2 = A1 @ W2 + b2             # (4, 1)
    A2 = sigmoid(Z2).flatten()     # (4,) — predictions
    
    loss = bce(y_xor, A2)
    losses_bp.append(loss)
    
    # ── BACKWARD PASS (chain rule) ──
    N = len(y_xor)
    
    # Output layer: dL/dA2
    dL_dA2 = -(y_xor / (A2 + 1e-9) - (1 - y_xor) / (1 - A2 + 1e-9)) / N  # (4,)
    
    # dL/dZ2 = dL/dA2 * sigmoid'(A2)
    dL_dZ2 = (dL_dA2 * sigmoid_deriv(A2)).reshape(-1, 1)  # (4, 1)
    
    # dL/dW2 = A1^T @ dL/dZ2
    dL_dW2 = A1.T @ dL_dZ2   # (4, 1)
    dL_db2 = dL_dZ2.sum(axis=0, keepdims=True)  # (1, 1)
    
    # Error at hidden layer: dL/dA1 = dL/dZ2 @ W2^T
    dL_dA1 = dL_dZ2 @ W2.T   # (4, 4)
    
    # dL/dZ1 = dL/dA1 * sigmoid'(A1)
    dL_dZ1 = dL_dA1 * sigmoid_deriv(A1)  # (4, 4)
    
    # dL/dW1 = X^T @ dL/dZ1
    dL_dW1 = X_xor.T @ dL_dZ1   # (2, 4)
    dL_db1 = dL_dZ1.sum(axis=0, keepdims=True)  # (1, 4)
    
    # ── UPDATE ──
    W2 -= lr * dL_dW2
    b2 -= lr * dL_db2
    W1 -= lr * dL_dW1
    b1 -= lr * dL_db1

# Final predictions
Z1 = X_xor @ W1 + b1
A1 = sigmoid(Z1)
A2_final = sigmoid(A1 @ W2 + b2).flatten()

print("=== XOR Problem Solved with 2-Layer Net (Backprop from scratch) ===")
for i, (x_s, y_s, p) in enumerate(zip(X_xor, y_xor, A2_final)):
    print(f"  Input {x_s.astype(int)} → True: {int(y_s)}, Predicted: {p:.4f}, Class: {int(p>0.5)}  {'✓' if int(p>0.5)==y_s else '✗'}")

plt.figure(figsize=(8, 4))
plt.plot(losses_bp, color='darkorange', lw=2)
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('Training Loss — 2-Layer Network on XOR (Pure Numpy Backprop)')
plt.show()

In [ ]:
# PyTorch version — autograd does the backprop automatically
print("=== Same XOR network using PyTorch autograd ===")

X_t = torch.tensor(X_xor, dtype=torch.float32)
y_t = torch.tensor(y_xor, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
criterion = nn.BCELoss()

for epoch in range(5000):
    optimizer.zero_grad()
    out = model(X_t).squeeze()
    loss = criterion(out, y_t)
    loss.backward()   # autograd computes all gradients (same chain rule, automated)
    optimizer.step()

with torch.no_grad():
    preds = model(X_t).squeeze()
    for i, (x_s, y_s, p) in enumerate(zip(X_xor, y_xor, preds)):
        print(f"  Input {x_s.astype(int)} → True: {int(y_s)}, Predicted: {p:.4f}  {'✓' if int(p>0.5)==y_s else '✗'}")

print("\n💡 PyTorch's .backward() implements backpropagation via autograd.")
print("   Every operation is tracked in a computation graph.")
print("   .backward() traverses this graph in reverse, applying the chain rule at each node.")

---
## 8. The Hessian & Second-Order Insight

In [ ]:
# Compute Hessian numerically and analyse critical points
def hessian_2d(f, x, y, h=1e-4):
    """Numerical Hessian of f(x,y) at (x,y)"""
    fxx = (f(x+h, y) - 2*f(x, y) + f(x-h, y)) / h**2
    fyy = (f(x, y+h) - 2*f(x, y) + f(x, y-h)) / h**2
    fxy = (f(x+h, y+h) - f(x+h, y-h) - f(x-h, y+h) + f(x-h, y-h)) / (4*h**2)
    return np.array([[fxx, fxy], [fxy, fyy]])

test_functions = [
    (lambda x, y: x**2 + y**2,        (0, 0), 'x²+y² → minimum at (0,0)'),
    (lambda x, y: -(x**2 + y**2),     (0, 0), '-x²-y² → maximum at (0,0)'),
    (lambda x, y: x**2 - y**2,        (0, 0), 'x²-y² → saddle at (0,0)'),
    (lambda x, y: (1-x)**2 + 100*(y-x**2)**2, (-1, 1), 'Rosenbrock → tricky valley'),
]

for f_fn, point, name in test_functions:
    x0, y0 = point
    H = hessian_2d(f_fn, x0, y0)
    eigvals = np.linalg.eigvals(H)
    
    if all(eigvals > 0):   kind = 'LOCAL MINIMUM'
    elif all(eigvals < 0): kind = 'LOCAL MAXIMUM'
    else:                  kind = 'SADDLE POINT'
    
    print(f"{name}")
    print(f"  Hessian eigenvalues: {eigvals.round(2)}")
    print(f"  Condition number: {abs(eigvals.max()/eigvals.min()):.1f}")
    print(f"  Type: {kind}")
    print()

---
## 9. Advanced Optimisers & Learning Rate Schedules

In [ ]:
# Learning rate schedules used in practice
epochs = 100
t = np.arange(epochs)

lr_max, lr_min = 0.1, 1e-4

# 1. Constant
lr_constant = np.ones(epochs) * 0.1

# 2. Step decay (halve every 25 epochs)
lr_step = 0.1 * (0.5 ** (t // 25))

# 3. Cosine annealing
lr_cosine = lr_min + 0.5*(lr_max - lr_min) * (1 + np.cos(np.pi * t / epochs))

# 4. Warmup + cosine (Transformer style)
warmup = 10
lr_warmup = np.where(t < warmup,
                     lr_max * t / warmup,
                     lr_min + 0.5*(lr_max-lr_min)*(1 + np.cos(np.pi*(t-warmup)/(epochs-warmup))))

# 5. 1-cycle (used with SGD in fastai)
half = epochs // 2
lr_1cycle = np.concatenate([
    np.linspace(lr_min, lr_max, half),
    np.linspace(lr_max, lr_min/100, epochs - half)
])

plt.figure(figsize=(12, 5))
for lr, name, col in zip(
    [lr_constant, lr_step, lr_cosine, lr_warmup, lr_1cycle],
    ['Constant', 'Step Decay', 'Cosine Annealing', 'Warmup+Cosine (Transformers)', '1-Cycle (fastai)'],
    ['gray', 'blue', 'darkorange', 'red', 'purple']
):
    plt.plot(lr, label=name, lw=2, color=col)

plt.xlabel('Epoch'); plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedules in Modern ML Training')
plt.legend()
plt.show()

print("💡 Warmup+Cosine: why Transformers need warmup")
print("   Early on: attention weights are random → gradients are large and noisy")
print("   Warmup lets the model settle before big update steps")
print("   Cosine decay smoothly brings LR to near-zero, fine-tuning convergence")

In [ ]:
# Gradient clipping — prevents exploding gradients
np.random.seed(42)

# Simulate gradient norms over training (occasionally spikes)
steps = 100
grads = np.abs(np.random.randn(steps)) * 0.5
grads[20] = 15.0   # simulate gradient explosion spike
grads[55] = 12.0
grads[80] = 18.0

clip_threshold = 5.0
clipped_grads = np.minimum(grads, clip_threshold)

plt.figure(figsize=(10, 4))
plt.plot(grads, 'r-', alpha=0.7, label='Raw gradient norm')
plt.plot(clipped_grads, 'b-', lw=2, label=f'After clipping (max={clip_threshold})')
plt.axhline(clip_threshold, color='k', linestyle='--', alpha=0.5, label='Clip threshold')
plt.xlabel('Training step'); plt.ylabel('Gradient L2 norm')
plt.title('Gradient Clipping — Prevents Exploding Gradients in RNNs/Transformers')
plt.legend()
plt.show()

print("Gradient clipping: if ||g||₂ > τ, scale g to have ||g||₂ = τ")
print("This preserves gradient direction but limits step size.")
print("Without it, one bad mini-batch can destroy hours of training.")

---
## 10. Full Training Loop — Putting It All Together

In [ ]:
# Train an MLP on a real dataset with all the ingredients:
# weight init, batch norm, Adam, cosine LR schedule, gradient clipping

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Data
wine = load_wine()
X_w, y_w = wine.data, wine.target
scaler = StandardScaler()
X_w = scaler.fit_transform(X_w)
X_tr, X_te, y_tr, y_te = train_test_split(X_w, y_w, test_size=0.2, random_state=42)

X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te, dtype=torch.float32)
y_te_t = torch.tensor(y_te, dtype=torch.long)

# Model with BatchNorm (which uses mean/variance — Statistics section!)
model_wine = nn.Sequential(
    nn.Linear(13, 64),
    nn.BatchNorm1d(64),    # normalises activations (Stats: mean, variance)
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Linear(32, 3)       # 3 wine classes
)

optimizer = torch.optim.Adam(model_wine.parameters(), lr=0.01, weight_decay=1e-4)  # L2 reg!
criterion = nn.CrossEntropyLoss()  # = softmax + negative log-likelihood (MLE!)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

train_losses, test_accs, lrs_used = [], [], []

for epoch in range(200):
    model_wine.train()
    optimizer.zero_grad()
    out = model_wine(X_tr_t)
    loss = criterion(out, y_tr_t)
    loss.backward()  # BACKPROP (chain rule)
    torch.nn.utils.clip_grad_norm_(model_wine.parameters(), max_norm=5.0)  # gradient clipping
    optimizer.step()
    scheduler.step()
    
    lrs_used.append(optimizer.param_groups[0]['lr'])
    train_losses.append(loss.item())
    
    model_wine.eval()
    with torch.no_grad():
        te_preds = model_wine(X_te_t).argmax(dim=1)
        test_accs.append((te_preds == y_te_t).float().mean().item())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(train_losses, 'darkorange', lw=1.5)
axes[0].set_title('Training Loss (Cross-Entropy = MLE!)'); axes[0].set_xlabel('Epoch')

axes[1].plot(test_accs, 'steelblue', lw=1.5)
axes[1].set_title(f'Test Accuracy (Final: {test_accs[-1]:.1%})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')

axes[2].plot(lrs_used, 'green', lw=1.5)
axes[2].set_title('Learning Rate Schedule (Cosine Annealing)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')

plt.tight_layout()
plt.show()

print(f"\n🎓 EVERYTHING CONNECTS:")
print(f"  📊 Statistics  → BatchNorm uses mean/variance; CrossEntropyLoss = MLE")
print(f"  🔢 Linear Algebra → Every layer = matrix multiply; weights are matrices")
print(f"  ∫  Calculus    → loss.backward() = chain rule; optimizer.step() = gradient descent")
print(f"\n  Final test accuracy: {test_accs[-1]:.1%} on Wine Classification 🍷")